# Busan Traffic AI PBL - 16_busan_pytorch_mlp.ipynb

부산시 교통량 예측 프로젝트 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
# =========================
# [셀 1] 라이브러리 로드
# =========================
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# =========================
# [셀 2] 데이터 로드 및 병합
# =========================
X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

y_candidates = [c for c in df_y.columns if c != "date"]
y_col = y_candidates[0] if len(y_candidates) == 1 else ( "y" if "y" in y_candidates else y_candidates[-1] )

df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)
df.head()

In [ ]:
# =========================
# [셀 3] split
# =========================
train_mask = (df["date"] >= "2022-01-01") & (df["date"] <= "2022-12-31")
val_mask   = (df["date"] >= "2023-01-01") & (df["date"] <= "2023-01-31")
test_mask  = (df["date"] >= "2023-02-01") & (df["date"] <= "2023-12-31")

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()


In [ ]:
# =========================
# [셀 4] 전처리
# =========================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import numpy as np

feature_cols = [c for c in df.columns if c not in ["date", y_col]]

X_train_raw, y_train = df_train[feature_cols], df_train[y_col].values.astype(np.float32)
X_val_raw,   y_val   = df_val[feature_cols],   df_val[y_col].values.astype(np.float32)
X_test_raw,  y_test  = df_test[feature_cols],  df_test[y_col].values.astype(np.float32)

# 숫자형 컬럼 선택 (휴일 is_holiday도 숫자면 자동 포함)
num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
print("사용되는 수치형 feature:", num_cols)

X_train_raw = X_train_raw[num_cols]
X_val_raw   = X_val_raw[num_cols]
X_test_raw  = X_test_raw[num_cols]

# X 전처리 파이프라인
x_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train = x_preprocess.fit_transform(X_train_raw).astype(np.float32)
X_val   = x_preprocess.transform(X_val_raw).astype(np.float32)
X_test  = x_preprocess.transform(X_test_raw).astype(np.float32)

# y 표준화 (학습 안정화 핵심)
y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).astype(np.float32)
y_val_s   = y_scaler.transform(y_val.reshape(-1, 1)).astype(np.float32)
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).astype(np.float32)

X_train.shape, X_val.shape, X_test.shape

In [ ]:
# =========================
# [셀 5] PyTorch Dataset/DataLoader (표준화된 y 사용)
# =========================
import torch
from torch.utils.data import Dataset, DataLoader

class TabDataset(Dataset):
    def __init__(self, X, y_scaled):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y_scaled).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
train_loader = DataLoader(TabDataset(X_train, y_train_s), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TabDataset(X_val,   y_val_s),   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TabDataset(X_test,  y_test_s),  batch_size=batch_size, shuffle=False)


In [ ]:
# =========================
# [셀 6] MLP 모델
# =========================
import torch.nn as nn

class MLPRegressor(nn.Module):
    def __init__(self, in_dim, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPRegressor(in_dim=X_train.shape[1], hidden=16).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# =========================
# [셀 7] 학습 루프 (val best 모델 저장)
# =========================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import copy

def eval_loader(model, loader, y_scaler):
    model.eval()
    preds_s, trues_s = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb)
            preds_s.append(pred.cpu().numpy())
            trues_s.append(yb.numpy())

    preds_s = np.vstack(preds_s).reshape(-1, 1)  # scaled
    trues_s = np.vstack(trues_s).reshape(-1, 1)  # scaled

    preds = y_scaler.inverse_transform(preds_s).ravel()
    trues = y_scaler.inverse_transform(trues_s).ravel()

    mae  = mean_absolute_error(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    r2   = r2_score(trues, preds)
    return mae, rmse, r2

epochs = 300

best_state = None
best_val_rmse = float("inf")
best_epoch = -1

for epoch in range(1, epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)  # scaled space에서 MSE
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0:
        val_mae, val_rmse, val_r2 = eval_loader(model, val_loader, y_scaler)
        print(f"epoch {epoch:03d} | val MAE {val_mae:.1f} RMSE {val_rmse:.1f} R2 {val_r2:.3f}")

        # val RMSE 기준 best 모델 갱신
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

print(f"BEST VAL @ epoch {best_epoch:03d} | RMSE {best_val_rmse:.1f}")

# (선택) best weight를 바로 모델에 적용해두기
model.load_state_dict(best_state)

In [ ]:
# =========================
# [셀 8] 최종 test 평가 (best 모델로)
# =========================#

test_mae, test_rmse, test_r2 = eval_loader(model, test_loader, y_scaler)
print(f"TEST (best@{best_epoch}): MAE {test_mae:.1f} RMSE {test_rmse:.1f} R2 {test_r2:.3f}")
